In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

The primary variables are in-vehicle time, other travel times, cost (the influence of which is derived from the automobile in-vehicle time coefficient and the persons’ modeled value of time), characteristics of the destination zone, demographics, and the household’s level of auto ownership.

In [3]:
import pandas as pd
import polars as pl
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [4]:
# read data
hh_data = util.get_hh_data(summary_config, uncloned=True)
per_data = util.get_person_data(summary_config, uncloned=True)
tour_data = util.get_tour_data(summary_config)

In [5]:
per_data = per_data.merge(hh_data[['household_id','auto_ownership','auto_ownership_4+','log_hh_1','log_hh_1_group',
                                   'auto_available_driver','auto_available_worker','source']],
                          how='left', on=['household_id','source']) # get auto ownership from hh data

tour_data = tour_data.merge(per_data, how='left', on=['person_id','household_id','source'])

## Total tour mode choice

In [6]:
# aggregate transit modes
transit_modes = ['WALK_LOC','WALK_COM','WALK_FRY','WALK_LR','DRIVE_TRN']
tour_mode_ordered = ["DRIVEALONEFREE", "SHARED2FREE", "SHARED3FREE", "BIKE","WALK","ALL_TRANSIT","SCH_BUS","TNC","Other"]
tour_data['tour_mode_transit_agg'] = tour_data['tour_mode'].apply(lambda x: "ALL_TRANSIT" if x in transit_modes else x)

df_plot = tour_data.groupby(['source','tour_mode_transit_agg'])['tour_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['tour_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot, x="tour_mode_transit_agg", y="percentage", color="source",barmode="group",
             category_orders={"tour_mode_transit_agg": tour_mode_ordered},
             title="Tour mode choice: all modes")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, yaxis=dict(tickformat=".1%"))
fig.show()

In [7]:
# show only transit modes
df_plot = tour_data.groupby(['source','tour_mode'])['tour_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['tour_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot.loc[df_plot['tour_mode'].isin(transit_modes)], x="tour_mode", y="percentage", color="source",barmode="group",
             title="Tour mode choice: transit modes")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, yaxis=dict(tickformat=".1%"))
fig.show()

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\3105823337.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Tour mode choice by segments

In [8]:
def plot_mode_choice(df: pd.DataFrame, grp_var: str, n_nol: int, height: int):
    df_plot = df.groupby(['source',grp_var,'tour_mode_transit_agg'])['tour_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['tour_weight']. \
        apply(lambda x: x / float(x.sum()))

    fig = px.bar(df_plot,
                 x="percentage", y="tour_mode_transit_agg", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 category_orders={"tour_mode_transit_agg": tour_mode_ordered},
                 title="Tour mode choice by " + grp_var)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.update_layout(height=height, width=700, xaxis1=dict(tickformat=".0%"), xaxis2=dict(tickformat=".0%")
                      )
    fig.show()
def plot_mode_choice_transit(df: pd.DataFrame,  grp_var: str, n_nol: int, height: int):
    df_plot = df.groupby(['source',grp_var,'tour_mode'])['tour_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['tour_weight']. \
        apply(lambda x: x / float(x.sum()))

    fig = px.bar(df_plot.loc[df_plot['tour_mode'].isin(transit_modes)],
                 x="percentage", y="tour_mode", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 category_orders={grp_var: pd.Series(df_plot[grp_var].unique()).sort_values().to_list()},
                 # color_discrete_sequence=config['psrc_color'],
                 title="Tour mode choice by " + grp_var + ": disaggregated transit modes")
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.update_layout(height=height, width=700, xaxis1=dict(tickformat=".1%"), xaxis2=dict(tickformat=".1%")
                      )
    fig.show()

In [10]:
plot_mode_choice(tour_data,'tour_type',3,1200)
plot_mode_choice_transit(tour_data,'tour_type',3,800)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [11]:
plot_mode_choice(tour_data,'ptype_label',2,1200)
plot_mode_choice_transit(tour_data,'ptype_label',3,800)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [12]:
plot_mode_choice(tour_data,'household_density_bin',3,600)
plot_mode_choice_transit(tour_data,'household_density_bin',3,500)

KeyError: 'household_density_bin'

In [13]:
plot_mode_choice(tour_data,'auto_ownership_simple',3,600)
plot_mode_choice_transit(tour_data,'auto_ownership_simple',3,500)

KeyError: 'auto_ownership_simple'

In [14]:
plot_mode_choice(tour_data,'auto_available_driver',2,600)
plot_mode_choice_transit(tour_data,'auto_available_driver',3,400)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [15]:
plot_mode_choice(tour_data,'auto_available_worker',2,600)
plot_mode_choice_transit(tour_data,'auto_available_worker',3,400)

C:\Users\Modeller\AppData\Local\Temp\ipykernel_36964\1430348015.py:16: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

